In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
VIDEO_STEM = "video_fixture"  # change to match your processed video
OUTPUT_DIR = Path("..") / VIDEO_STEM

In [ ]:
# ── Load output ───────────────────────────────────────────────────────────────
segments = json.loads((OUTPUT_DIR / "segments.json").read_text())
frame_metrics = json.loads((OUTPUT_DIR / "frame_metrics.json").read_text())

print(f"Loaded {len(segments)} segments, {len(frame_metrics)} frame metrics")

In [ ]:
# ── Segment overview table ────────────────────────────────────────────────────
seg_rows = []
for s in segments:
    seg_rows.append({
        "segment_id": s["segment_id"],
        "start_time": s["start_time"],
        "end_time": s["end_time"],
        "duration": round(s["end_time"] - s["start_time"], 2),
        "n_camera_movements": len(s["camera_movements"]),
    })

df_segs = pd.DataFrame(seg_rows)
display(df_segs)

In [ ]:
# ── Camera movements flat table ───────────────────────────────────────────────
move_rows = []
for s in segments:
    for m in s["camera_movements"]:
        move_rows.append(m)

df_moves = pd.DataFrame(move_rows)
display(df_moves)

In [ ]:
# ── Signal plot: pan/tilt/zoom over time with boundaries ──────────────────────
df_fm = pd.DataFrame(frame_metrics).dropna(subset=["pan", "tilt", "zoom"])

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
channels = ["pan", "tilt", "zoom"]
colors = ["steelblue", "darkorange", "forestgreen"]

for ax, ch, col in zip(axes, channels, colors):
    ax.plot(df_fm["timestamp"], df_fm[ch], color=col, linewidth=0.8)
    ax.set_ylabel(ch)

    # Scene boundaries (thick red lines)
    for s in segments:
        ax.axvline(s["start_time"], color="red", linewidth=1.5, alpha=0.7)

    # Movement boundaries (thin dashed grey)
    for s in segments:
        for m in s["camera_movements"]:
            ax.axvline(m["start_time"], color="grey", linewidth=0.7, linestyle="--", alpha=0.5)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Camera motion signal with scene (red) and movement (grey) boundaries")
scene_patch = mpatches.Patch(color="red", label="Scene boundary")
move_patch = mpatches.Patch(color="grey", label="Movement boundary")
fig.legend(handles=[scene_patch, move_patch], loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Velocity profile: entry vs exit per channel ───────────────────────────────
if len(df_moves) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, ch in zip(axes, ["pan", "tilt", "zoom"]):
        ev = df_moves[f"{ch}_entry_vel"]
        xv = df_moves[f"{ch}_exit_vel"]
        sc = ax.scatter(ev, xv, c=df_moves["segment_id"], cmap="tab10", alpha=0.8)
        ax.axline((0, 0), slope=1, color="grey", linewidth=0.8, linestyle="--")
        ax.axhline(0, color="black", linewidth=0.4)
        ax.axvline(0, color="black", linewidth=0.4)
        ax.set_xlabel("entry_vel")
        ax.set_ylabel("exit_vel")
        ax.set_title(ch)
        plt.colorbar(sc, ax=ax, label="segment_id")
    fig.suptitle("Entry vs Exit velocity per channel (coloured by scene)")
    plt.tight_layout()
    plt.show()
else:
    print("No camera movements found.")